# iFLiP lifetime and synchronization QC

Start here for every session. This notebook locates paired files, calculates background-corrected MPET, aligns iFLiP to NI-DAQ, and displays acquisition QC. Edit only the configuration cell, then run all cells. Use a background acquired at the matching laser power and detector settings.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from iflip3 import process_aligned_session, read_iflip3, session_paths

## Session configuration

In [ ]:
MOUSE = "MOUSE_ID"
DATE = "YYMMDD"
RUN = 1
DATA_ROOT = Path("Z:/")
BACKGROUND_FILE = Path(r"Z:\FLIM FLIP\backgrounds\matched_background.iFLiP3")
AFTERPULSE_RATIO = 0.03
SPC_RANGE_NS = (0.4, 12.3)

In [ ]:
if MOUSE == "MOUSE_ID" or DATE == "YYMMDD":
    raise ValueError("Set MOUSE and DATE in the configuration cell first.")
paths = session_paths(MOUSE, DATE, RUN, data_root=DATA_ROOT)
print(f"iFLiP:      {paths.iflip}")
print(f"NI-DAQ:     {paths.nidaq}")
print(f"Running:    {paths.running}")
print(f"Background: {BACKGROUND_FILE}")
paths.validate()
if not BACKGROUND_FILE.is_file():
    raise FileNotFoundError(f"Background file not found: {BACKGROUND_FILE}")

## Process and align the session

In [ ]:
session = process_aligned_session(
    paths.iflip, paths.nidaq, BACKGROUND_FILE, running_path=paths.running,
    spc_range=SPC_RANGE_NS, afterpulse_ratio=AFTERPULSE_RATIO,
)
print(f"Lifetime samples: {session.mpet_ns.size}")
print(f"Matched sync pulses: {session.alignment.matched_pulses}")
print(f"Clock drift: {session.alignment.drift_ppm:.3f} ppm")
print(f"Sync residual RMS: {1000 * session.alignment.rms_residual_seconds:.3f} ms")
print(f"Maximum sync residual: {1000 * session.alignment.max_residual_seconds:.3f} ms")
print(f"Licks / cues / Ensure: {session.lick_times_nidaq.size} / {session.visual_cue_pulses.onset_times.size} / {session.ensure_pulses.onset_times.size}")

## Aggregate photon-arrival histogram

Use this raw aggregate view to identify abnormal peaks or truncated decays. MPET below is calculated from corrected data.

In [ ]:
recording = read_iflip3(paths.iflip)
aggregate = recording.data[:, :, 0].sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(recording.lifetime_time, aggregate, "o-", ms=3)
ax.axvspan(*SPC_RANGE_NS, color="tab:blue", alpha=0.08, label="MPET window")
ax.set(xlabel="Photon-arrival time (ns)", ylabel="Aggregated photon counts")
ax.legend(); plt.show()

## Lifetime, intensity QC, and clock residuals

MPET is the primary signal. Intensity remains a QC channel because it can contain biological changes and artifacts. Clock residuals assess alignment; lifetime samples remain spaced at 0.1 seconds for a 10 Hz recording.

In [ ]:
time = session.lifetime_time_nidaq
baseline = time < time[0] + min(50.0, np.ptp(time) / 6)
delta_mpet = session.mpet_ns - np.nanmedian(session.mpet_ns[baseline])
f0 = np.nanmedian(session.raw_intensity_counts[baseline])
dff = 100 * (session.raw_intensity_counts / f0 - 1)
fig, axes = plt.subplots(3, 1, figsize=(12, 8), constrained_layout=True)
axes[0].plot(time, delta_mpet); axes[0].set_ylabel("Delta MPET (ns)")
axes[1].plot(time, dff, color="0.4"); axes[1].set_ylabel("Intensity dF/F (%)")
axes[2].plot(session.alignment.iflip_marker_times, 1000 * session.alignment.residuals_seconds, ".", ms=3)
axes[2].set(xlabel="iFLiP time (s)", ylabel="Clock residual (ms)")
for ax in axes: ax.axhline(0, color="black", lw=0.8)
for onset, offset in zip(session.ensure_pulses.onset_times, session.ensure_pulses.offset_times, strict=True):
    axes[0].axvspan(onset, offset, color="tab:orange", alpha=0.2)
    axes[1].axvspan(onset, offset, color="tab:orange", alpha=0.2)
plt.show()

### QC checklist

- Confirm a smooth aggregate decay with the expected peak and tail.
- Confirm clock residuals stay near zero without jumps or curvature.
- Confirm plausible lick, cue, and Ensure counts.
- Use MPET as the primary lifetime readout and intensity as QC.
- Record the background filename and afterpulse ratio.